# Indiana — Title 27 (Insurance) → `data/indiana/ins_codes/*.md`

Indiana’s insurance statutes are codified in **Indiana Code Title 27 — Insurance** (official portal: [iga.in.gov](https://iga.in.gov/legislative/laws/2024/ic/)). That official site is a **SPA** and is not convenient to bulk-download with plain HTML requests.

This notebook mirrors **Justia’s** browse tree: **[Indiana Code — Title 27](https://law.justia.com/codes/indiana/title-27/)** (`/codes/indiana/title-27/…`). Links are organized under **article → chapter → section** and each section has a **`section-27-…`** URL (for example: `…/section-27-1-1-5-1/`).

**Cloudflare** often blocks plain **`httpx`** for Justia. We use **`curl_cffi`** with **`impersonate="chrome120"`** (same approach as `ins_ipynb/idaho.ipynb`).

**Discovery:** BFS from the Title 27 index, following only paths under **`/codes/indiana/title-27/`** that are not **`/section-…`** pages; every **`/section-…`** link is collected (~3–4k sections).

**Download:** Each section page’s text comes from **`div.primary-content`**, with light Justia boilerplate stripped. Files are **`IN_sec_<section>.md`** where `<section>` is the `section-` slug with hyphens replaced by underscores (e.g. `IN_sec_27_1_1_5_1.md`).

**Config:** `CODE_YEAR` appears in the seed URL (`/codes/indiana/{year}/title-27/`, often redirects to the canonical tree). `MAX_SECTIONS` caps downloads (0 = all). `MAX_DISCOVERY_PAGES` caps index pages fetched during discovery (0 = no cap). `REUSE_DISCOVERED_URLS` skips discovery when `_indiana_title27_section_urls.txt` exists.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/indiana/title-27"
CODE_YEAR = "2024"
TITLE_INDEX = f"{BASE}/codes/indiana/{CODE_YEAR}/title-27/"

OUT_DIR = Path("data") / "indiana" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_indiana_title27_section_urls.txt"
REUSE_DISCOVERED_URLS = True

section_label_re = re.compile(r"/section-(27-[^/]+)/?$", re.I)


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def discover_section_urls() -> list[str]:
    """BFS article/chapter pages; collect section URLs under Title 27."""
    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start)}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0

    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu)
            if not p.startswith(PATH_PREFIX):
                continue
            if "/section-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")

    return sorted(sections, key=lambda u: section_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    m = section_label_re.search(urlparse(url).path)
    if not m:
        raise ValueError(f"cannot parse section id from {url!r}")
    return m.group(1)


def section_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"IN_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    drop_exact = {
        "of",
        "this Section",
        "Universal Citation:",
        "Next",
        "Previous",
    }
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if s in drop_exact:
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Indiana Code" in s and "(" in s:
            continue
        if s.startswith("IN Code §"):
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        if s.startswith("Indiana") and "more current" in s:
            continue
        if s.startswith("may have more current"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title27() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: section_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Title 27")
        all_urls = found
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"Indiana Code § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Indiana Code — Title 27 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [iga.in.gov — Indiana Code](https://iga.in.gov/legislative/laws/{CODE_YEAR}/ic/)\n\n"
                    f"**Section:** §{label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title27()


Discovered 3710 section URLs under Title 27
… 200/3710 (wrote=198 skipped=2 failed=0)
… 400/3710 (wrote=397 skipped=3 failed=0)
… 600/3710 (wrote=597 skipped=3 failed=0)
… 800/3710 (wrote=794 skipped=6 failed=0)
… 1000/3710 (wrote=994 skipped=6 failed=0)
… 1200/3710 (wrote=1189 skipped=11 failed=0)
… 1400/3710 (wrote=1389 skipped=11 failed=0)
… 1600/3710 (wrote=1589 skipped=11 failed=0)
… 1800/3710 (wrote=1787 skipped=13 failed=0)
… 2000/3710 (wrote=1987 skipped=13 failed=0)
… 2200/3710 (wrote=2186 skipped=14 failed=0)
… 2400/3710 (wrote=2385 skipped=15 failed=0)
… 2600/3710 (wrote=2585 skipped=15 failed=0)
… 2800/3710 (wrote=2785 skipped=15 failed=0)
… 3000/3710 (wrote=2985 skipped=15 failed=0)
… 3200/3710 (wrote=3185 skipped=15 failed=0)
… 3400/3710 (wrote=3384 skipped=16 failed=0)
… 3600/3710 (wrote=3584 skipped=16 failed=0)
Done. wrote=3694 skipped=16 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/indiana/ins_codes


{'wrote': 3694, 'skipped': 16, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
